<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;"><div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div><div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div></div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Image Transformation — Algorithm</b></h1></div>


## Purpose and reading order

This notebook is the algorithm specification for the corresponding `main.ipynb` in this lab. It has two linked layers:

| Layer | Content | Question answered |
| --- | --- | --- |
| **Part I — Complete pseudocode** | One self-contained executable-style block with explicit branches and gates | What is the whole pipeline? |
| **Part II — Auditable specification** | Six-section contract for every step: Purpose, Inputs, Procedure, Produces, Acceptance, Failure mode | Why is each step valid? |

Names, shapes, thresholds, and failure decisions are kept identical between both layers. The acceptance contract of step *k* is the precondition of step *k+1*.

## Part I — Complete pseudocode

The single block below is the complete pipeline. Every named artifact is assigned before use; every failure is explicit. It is a direct, implementation-neutral transcription of the executable notebook.

```text
ALGORITHM Image_Transformation

# Algorithm 1 — Setup — Environment and Configuration
PLOT.rcParams["figure.dpi"] ← 110
PLOT.rcParams["axes.titlesize"] ← 11

# Algorithm 2 — 1. Data and Output Paths
PROCEDURE locate_lab_root(start)
    start ← start.resolve()
    FOR candidate IN [start, EXPAND start.parents] DO
        IF ((candidate / "data").is_dir() AND (candidate / "notebooks").is_dir()) THEN
            RETURN candidate
        END IF
    END FOR
    RAISE FileNotFoundError("Could not locate the image-transformation lab root.")
END PROCEDURE
LAB_DIR ← locate_lab_root(PATH.cwd())
DATA_DIR ← (LAB_DIR / "data")
OUTPUT_DIR ← ((LAB_DIR / "outputs") / "figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
IMAGE_EXTENSIONS ← {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES ← SORT(STREAM[path FOR path IN DATA_DIR.rglob("*") WHERE (path.is_file() AND (path.suffix.lower() IN IMAGE_EXTENSIONS))])
REQUIRE DATA_FILES OTHERWISE FORMAT("No supported images found in: {0}" , DATA_DIR)
PROCEDURE select_input_image(VARIADIC keywords)
    keywords ← TUPLE(STREAM[keyword.lower() FOR keyword IN keywords])
    matches ← LIST[path FOR path IN DATA_FILES WHERE ALL(STREAM[(keyword IN path.stem.lower()) FOR keyword IN keywords])]
    IF (NOT matches) THEN
        RAISE FileNotFoundError(FORMAT("No image matching {0} found in {1}" , keywords, DATA_DIR))
    END IF
    RETURN matches[0]
END PROCEDURE
ROLE_PATTERNS ← {"ascent" ↦ ("ascent",), "ballons" ↦ ("ballon",), "einstein" ↦ ("einstein",), "tower" ↦ ("tower",
    ), "peppers" ↦ ("pepper",)}
IMAGE_FILES ← MAP{role ↦ select_input_image(EXPAND patterns) FOR (role, patterns) IN ITEMS(ROLE_PATTERNS)}
CALL EMIT("Lab directory :", LAB_DIR)
CALL EMIT("Data directory:", DATA_DIR)
CALL EMIT("Output folder :", OUTPUT_DIR)
CALL EMIT("Images discovered:", LENGTH(DATA_FILES))
CALL EMIT("Experiment roles :", LENGTH(IMAGE_FILES))

# Algorithm 3 — 2. Load and Inspect the Reference Images
rgb_images ← MAP{name ↦ ARRAY.asarray(IMAGE.open(path).convert("RGB")) FOR (name,
    path) IN ITEMS(IMAGE_FILES)}
grayscale_images ← MAP{name ↦ ARRAY.asarray(IMAGE.open(path).convert("L")) FOR (name,
    path) IN ITEMS(IMAGE_FILES)}
FOR name IN IMAGE_FILES DO
    rgb ← rgb_images[name]
    gray ← grayscale_images[name]
    CALL EMIT(FORMAT("{0:9s} | RGB={1:16s} gray={2:12s} dtype={3} range=[{4}, {5}]" , name,
        STRING(rgb.shape), STRING(gray.shape), gray.dtype, MIN(gray), MAX(gray)))
END FOR
(fig, axes) ← PLOT.subplots(1, LENGTH(rgb_images), figsize = (16, 4))
FOR (ax, (name, image)) IN ZIP(axes, ITEMS(rgb_images)) DO
    CALL ax.imshow(image)
    CALL ax.set_title(FORMAT("{0}\n{1}×{2}" , name, image.shape[1], image.shape[0]))
    CALL ax.axis("off")
END FOR
CALL fig.suptitle("Reference Images")
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "01_reference_images.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 4 — 3. Intensity Transformation Model
ascent ← grayscale_images["ascent"]
identity ← COPY(ascent)
REQUIRE ARRAY.array_equal(identity, ascent) OTHERWISE STOP WITH ValidationError
CALL EMIT("Identity transformation preserves every pixel:", ARRAY.array_equal(identity, ascent))

# Algorithm 5 — 4. Image Negative
negative ← (255 − ascent)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
CALL axes[0].imshow(ascent, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(negative, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Negative")
CALL axes[2].plot(ARRAY.arange(256), (255 − ARRAY.arange(256)))
CALL axes[2].set_title("Transformation: s = 255 - r")
CALL axes[2].set_xlabel("Input intensity r")
CALL axes[2].set_ylabel("Output intensity s")
CALL axes[2].set_xlim(0, 255)
CALL axes[2].set_ylim(0, 255)
CALL axes[2].grid(alpha = 0.25)
FOR ax IN axes[:2] DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "02_negative.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 6 — 5. Brightness and Contrast
PROCEDURE apply_linear_intensity_transform(image, gain = 1.0, offset = 0.0)
    transformed ← ((gain * CAST(image, ARRAY.float32)) + offset)
    RETURN CAST(ARRAY.clip(transformed, 0, 255), ARRAY.uint8)
END PROCEDURE
brighter ← apply_linear_intensity_transform(ascent, gain = 1.0, offset = 50)
darker ← apply_linear_intensity_transform(ascent, gain = 1.0, offset = (−50))
higher_contrast ← apply_linear_intensity_transform(ascent, gain = 1.5, offset = (−64))
(fig, axes) ← PLOT.subplots(1, 4, figsize = (14, 4))
examples ← [("Original", ascent), ("Brightness +50", brighter), ("Brightness -50", darker), ("Higher contrast",
    higher_contrast)]
FOR (ax, (title, image)) IN ZIP(axes, examples) DO
    CALL ax.imshow(image, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "03_brightness_contrast.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 7 — 6. Contrast Stretching
PROCEDURE stretch_image_contrast(image)
    image_float ← CAST(image, ARRAY.float32)
    r_min ← MIN(image_float)
    r_max ← MAX(image_float)
    IF (r_max = r_min) THEN
        RETURN ARRAY.zeros_like(image)
    END IF
    stretched ← ((image_float − r_min) / (r_max − r_min))
    stretched ← (stretched * 255.0)
    RETURN CAST(ARRAY.clip(stretched, 0, 255), ARRAY.uint8)
END PROCEDURE
low_contrast_image ← (90 + ((CAST(ascent, ARRAY.float32) / 255.0) * 75))
low_contrast_image ← CAST(ARRAY.clip(low_contrast_image, 0, 255), ARRAY.uint8)
stretched ← stretch_image_contrast(low_contrast_image)
(fig, axes) ← PLOT.subplots(2, 2, figsize = (10, 7))
CALL axes[(0, 0)].imshow(low_contrast_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[(0, 0)].set_title("Low-contrast image")
CALL axes[(0, 0)].axis("off")
CALL axes[(0, 1)].hist(FLATTEN(low_contrast_image), bins = 256, range = (0, 256))
CALL axes[(0, 1)].set_title("Before stretching")
CALL axes[(0, 1)].set_xlabel("Intensity")
CALL axes[(1, 0)].imshow(stretched, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[(1, 0)].set_title("Contrast stretched")
CALL axes[(1, 0)].axis("off")
CALL axes[(1, 1)].hist(FLATTEN(stretched), bins = 256, range = (0, 256))
CALL axes[(1, 1)].set_title("After stretching")
CALL axes[(1, 1)].set_xlabel("Intensity")
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "04_contrast_stretching.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Before:", INTEGER(MIN(low_contrast_image)), "to", INTEGER(MAX(low_contrast_image)))
CALL EMIT("After :", INTEGER(MIN(stretched)), "to", INTEGER(MAX(stretched)))

# Algorithm 8 — 7. Logarithmic Transformation
PROCEDURE apply_log_intensity_transform(image)
    image_float ← CAST(image, ARRAY.float32)
    c ← (255.0 / ARRAY.log1p(255.0))
    transformed ← (c * ARRAY.log1p(image_float))
    RETURN CAST(ARRAY.clip(transformed, 0, 255), ARRAY.uint8)
END PROCEDURE
logged ← apply_log_intensity_transform(ascent)
r ← ARRAY.arange(256, dtype = ARRAY.float32)
log_response_curve ← ((255.0 / ARRAY.log1p(255.0)) * ARRAY.log1p(r))
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
CALL axes[0].imshow(ascent, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[0].axis("off")
CALL axes[1].imshow(logged, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Log transform")
CALL axes[1].axis("off")
CALL axes[2].plot(r, log_response_curve)
CALL axes[2].set_title("Log transformation curve")
CALL axes[2].set_xlabel("Input intensity r")
CALL axes[2].set_ylabel("Output intensity s")
CALL axes[2].grid(alpha = 0.25)
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "05_log_transform.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 9 — 8. Gamma / Power-Law Transformation
PROCEDURE apply_gamma_intensity_transform(image, gamma)
    IF (gamma ≤ 0) THEN
        RAISE ValueError("gamma must be strictly positive.")
    END IF
    normalized ← (CAST(image, ARRAY.float32) / 255.0)
    transformed ← (normalized ^ gamma)
    RETURN CAST(ARRAY.clip((transformed * 255.0), 0, 255), ARRAY.uint8)
END PROCEDURE
gamma_values ← [0.4, 0.7, 1.0, 1.5, 2.2]
(fig, axes) ← PLOT.subplots(1, LENGTH(gamma_values), figsize = (16, 3.6))
FOR (ax, gamma) IN ZIP(axes, gamma_values) DO
    transformed ← apply_gamma_intensity_transform(ascent, gamma)
    CALL ax.imshow(transformed, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(FORMAT("γ = {0}" , gamma))
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "06_gamma_examples.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
r ← ARRAY.linspace(0, 1, 256)
(fig, ax) ← PLOT.subplots(figsize = (6, 4.5))
FOR gamma IN gamma_values DO
    CALL ax.plot(r, (r ^ gamma), label = FORMAT("γ={0}" , gamma))
END FOR
CALL ax.plot(r, r, linestyle = "--", label = "identity")
CALL ax.set_title("Gamma / Power-Law Curves")
CALL ax.set_xlabel("Normalized input r")
CALL ax.set_ylabel("Normalized output s")
CALL ax.grid(alpha = 0.25)
CALL ax.legend()
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "07_gamma_curves.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 10 — 9. Histogram Equalization
PROCEDURE equalize_image_histogram(image)
    intensity_histogram ← ARRAY.bincount(FLATTEN(image), minlength = 256)
    intensity_probability ← (intensity_histogram / image.size)
    cumulative_distribution ← ARRAY.cumsum(intensity_probability)
    equalization_mapping ← CAST(ARRAY.round((255 * cumulative_distribution)), ARRAY.uint8)
    equalized_image ← equalization_mapping[image]
    RETURN (equalized_image, equalization_mapping)
END PROCEDURE
(equalized_image, equalization_map) ← equalize_image_histogram(low_contrast_image)
(fig, axes) ← PLOT.subplots(2, 2, figsize = (10, 7))
CALL axes[(0, 0)].imshow(low_contrast_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[(0, 0)].set_title("Before equalization")
CALL axes[(0, 0)].axis("off")
CALL axes[(0, 1)].hist(FLATTEN(low_contrast_image), bins = 256, range = (0, 256))
CALL axes[(0, 1)].set_title("Original histogram")
CALL axes[(1, 0)].imshow(equalized_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[(1, 0)].set_title("After equalization")
CALL axes[(1, 0)].axis("off")
CALL axes[(1, 1)].hist(FLATTEN(equalized_image), bins = 256, range = (0, 256))
CALL axes[(1, 1)].set_title("Equalized histogram")
FOR ax IN [axes[(0, 1)], axes[(1, 1)]] DO
    CALL ax.set_xlabel("Intensity")
    CALL ax.set_ylabel("Pixel count")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "08_histogram_equalization.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
(fig, ax) ← PLOT.subplots(figsize = (6, 4.5))
CALL ax.plot(ARRAY.arange(256), equalization_map)
CALL ax.set_title("Histogram Equalization Mapping")
CALL ax.set_xlabel("Input intensity")
CALL ax.set_ylabel("Mapped intensity")
CALL ax.set_xlim(0, 255)
CALL ax.set_ylim(0, 255)
CALL ax.grid(alpha = 0.25)
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "09_equalization_mapping.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 11 — 10. Compare the Fundamental Intensity Transformations
comparison ← [("Original", ascent), ("Negative", negative), ("Brighter", brighter), ("Contrast stretch",
    stretch_image_contrast(ascent)), ("Log", logged), ("Gamma 0.5", apply_gamma_intensity_transform(ascent,
    0.5)), ("Gamma 2.0", apply_gamma_intensity_transform(ascent, 2.0)), ("Hist. equalized",
    equalize_image_histogram(ascent)[0])]
(fig, axes) ← PLOT.subplots(2, 4, figsize = (14, 7))
FOR (ax, (title, image)) IN ZIP(FLATTEN(axes), comparison) DO
    CALL ax.imshow(image, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "10_intensity_transform_comparison.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 12 — 11. Geometric Transformation Model
PROCEDURE map_points_with_scale_and_shift(points, scale_x = 1.15, scale_y = 0.85, shift_x = 35.0,
    shift_y = 20.0)
    points ← ARRAY.asarray(points, dtype = ARRAY.float64)
    IF ((points.ndim ≠ 2) OR (points.shape[1] ≠ 2)) THEN
        RAISE ValueError("points must have shape (N, 2).")
    END IF
    x ← points[(:, 0)]
    y ← points[(:, 1)]
    x_prime ← ((scale_x * x) + shift_x)
    y_prime ← ((scale_y * y) + shift_y)
    RETURN ARRAY.column_stack((x_prime, y_prime))
END PROCEDURE
source_points ← ARRAY.array([[0.0, 0.0], [120.0, 0.0], [120.0, 80.0], [0.0, 80.0]], dtype = ARRAY.float64)
mapped_points ← map_points_with_scale_and_shift(source_points)
expected_mapped_points ← ARRAY.array([[35.0, 20.0], [173.0, 20.0], [173.0, 88.0], [35.0, 88.0]],
    dtype = ARRAY.float64)
REQUIRE (mapped_points.shape = source_points.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.all(ARRAY.isfinite(mapped_points)) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.allclose(mapped_points, expected_mapped_points) OTHERWISE STOP WITH ValidationError
CALL EMIT("Original coordinates:")
CALL EMIT(source_points)
CALL EMIT("\nTransformed coordinates:")
CALL EMIT(ARRAY.round(mapped_points, 2))
(fig, ax) ← PLOT.subplots(figsize = (6, 5))
closed_source_polygon ← ARRAY.vstack((source_points, source_points[0]))
closed_mapped_polygon ← ARRAY.vstack((mapped_points, mapped_points[0]))
CALL ax.plot(closed_source_polygon[(:, 0)], closed_source_polygon[(:, 1)], "o-",
    label = "Original coordinates")
CALL ax.plot(closed_mapped_polygon[(:, 0)], closed_mapped_polygon[(:, 1)], "o-",
    label = "Mapped coordinates")
CALL ax.set_title("Direct Coordinate Mapping: (x, y) → (x′, y′)")
CALL ax.set_xlabel("x")
CALL ax.set_ylabel("y")
CALL ax.set_aspect("equal", adjustable = "box")
CALL ax.grid(alpha = 0.25)
CALL ax.legend()
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "10_geometric_coordinate_model.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 13 — 12. Homogeneous Coordinates
PROCEDURE transform_point_with_matrix(matrix, x, y)
    point ← ARRAY.array([x, y, 1.0], dtype = ARRAY.float64)
    transformed ← (matrix · point)
    RETURN (transformed[:2] / transformed[2])
END PROCEDURE
identity_transform ← ARRAY.eye(3)
CALL EMIT("Identity matrix:")
CALL EMIT(identity_transform)
CALL EMIT("Point (10, 20) ->", transform_point_with_matrix(identity_transform, 10, 20))

# Algorithm 14 — 13. Fundamental Geometric Transformation Matrices
PROCEDURE build_translation_matrix(tx, ty)
    RETURN ARRAY.array([[1.0, 0.0, tx], [0.0, 1.0, ty], [0.0, 0.0, 1.0]])
END PROCEDURE
PROCEDURE build_scaling_matrix(sx, sy)
    RETURN ARRAY.array([[sx, 0.0, 0.0], [0.0, sy, 0.0], [0.0, 0.0, 1.0]])
END PROCEDURE
PROCEDURE build_rotation_matrix(angle_degrees)
    angle ← ARRAY.deg2rad(angle_degrees)
    c ← ARRAY.cos(angle)
    s ← ARRAY.sin(angle)
    RETURN ARRAY.array([[c, (−s), 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
END PROCEDURE
PROCEDURE build_shear_matrix(kx = 0.0, ky = 0.0)
    RETURN ARRAY.array([[1.0, kx, 0.0], [ky, 1.0, 0.0], [0.0, 0.0, 1.0]])
END PROCEDURE
PROCEDURE build_horizontal_reflection_matrix()
    RETURN ARRAY.array([[(−1.0), 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
END PROCEDURE

# Algorithm 15 — 14. Origin-Centered vs Centered Geometry
PROCEDURE center_transform_on_image(matrix, image_shape)
    (height, width) ← image_shape[:2]
    center_x ← ((width − 1) / 2)
    center_y ← ((height − 1) / 2)
    translate_to_origin ← build_translation_matrix((−center_x), (−center_y))
    translate_from_origin ← build_translation_matrix(center_x, center_y)
    RETURN ((translate_from_origin · matrix) · translate_to_origin)
END PROCEDURE

# Algorithm 16 — 15. Forward Mapping vs Inverse Mapping
PROCEDURE warp_image_affine(image, forward_matrix, output_shape = NULL, interpolation_order = 1,
    fill_value = 0.0)
    IF (output_shape IS NULL) THEN
        output_shape ← image.shape[:2]
    END IF
    (out_h, out_w) ← output_shape
    (yy, xx) ← ARRAY.indices((out_h, out_w), dtype = ARRAY.float64)
    output_homogeneous_coordinates ← ARRAY.stack([FLATTEN(xx), FLATTEN(yy), ARRAY.ones(xx.size)], axis = 0)
    inverse_transform_matrix ← ARRAY.linalg.inv(forward_matrix)
    input_homogeneous_coordinates ← (inverse_transform_matrix · output_homogeneous_coordinates)
    x_in ← input_homogeneous_coordinates[0]
    y_in ← input_homogeneous_coordinates[1]
    sample_coordinates ← ARRAY.vstack([y_in, x_in])
    IF (image.ndim = 2) THEN
        warped ← RESHAPE(INTERPOLATE_COORDINATES(CAST(image, ARRAY.float32), sample_coordinates,
            order = interpolation_order, mode = "constant", cval = fill_value), out_h, out_w)
    ELSE
        IF (image.ndim = 3) THEN
            warped_channels ← []
            FOR channel_index IN RANGE(image.shape[2]) DO
                sampled ← RESHAPE(INTERPOLATE_COORDINATES(CAST(image[(…, channel_index)], ARRAY.float32),
                    sample_coordinates, order = interpolation_order, mode = "constant", cval = fill_value),
                    out_h, out_w)
                CALL APPEND(warped_channels, sampled)
            END FOR
            warped ← ARRAY.stack(warped_channels, axis = (−1))
        ELSE
            RAISE ValueError("Expected a 2-D grayscale or 3-D color image.")
        END IF
    END IF
    RETURN CAST(ARRAY.clip(warped, 0, 255), ARRAY.uint8)
END PROCEDURE

# Algorithm 17 — 16. Translation
einstein ← grayscale_images["einstein"]
T ← build_translation_matrix(tx = 70, ty = 35)
translated_image ← warp_image_affine(einstein, T, interpolation_order = 0)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 4.5))
CALL axes[0].imshow(einstein, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(translated_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Translation: tx=70, ty=35")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "11_translation.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 18 — 17. Rotation
R_origin ← build_rotation_matrix(30)
R_center ← center_transform_on_image(build_rotation_matrix(30), einstein.shape)
rotated_origin ← warp_image_affine(einstein, R_origin, interpolation_order = 1)
rotated_center ← warp_image_affine(einstein, R_center, interpolation_order = 1)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (13, 4.2))
CALL axes[0].imshow(einstein, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(rotated_origin, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Rotation about origin")
CALL axes[2].imshow(rotated_center, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("Rotation about image center")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "12_rotation_origin_center.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 19 — 18. Scaling and Resizing
S_uniform ← center_transform_on_image(build_scaling_matrix(0.65, 0.65), einstein.shape)
S_nonuniform ← center_transform_on_image(build_scaling_matrix(1.35, 0.65), einstein.shape)
scaled_uniform ← warp_image_affine(einstein, S_uniform, interpolation_order = 1)
scaled_nonuniform ← warp_image_affine(einstein, S_nonuniform, interpolation_order = 1)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (13, 4.2))
CALL axes[0].imshow(einstein, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(scaled_uniform, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Uniform scale")
CALL axes[2].imshow(scaled_nonuniform, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("Non-uniform scale")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "13_scaling.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 20 — 19. Interpolation Comparison
peppers_rgb ← rgb_images["peppers"]
scale_for_interpolation ← center_transform_on_image(build_scaling_matrix(1.7, 1.7), peppers_rgb.shape)
nearest ← warp_image_affine(peppers_rgb, scale_for_interpolation, interpolation_order = 0)
bilinear ← warp_image_affine(peppers_rgb, scale_for_interpolation, interpolation_order = 1)
bicubic ← warp_image_affine(peppers_rgb, scale_for_interpolation, interpolation_order = 3)
(h, w) ← peppers_rgb.shape[:2]
crop ← (slice((h DIV 3), ((2 * h) DIV 3)), slice((w DIV 3), ((2 * w) DIV 3)))
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
CALL axes[0].imshow(peppers_rgb)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(nearest[crop])
CALL axes[1].set_title("Nearest")
CALL axes[2].imshow(bilinear[crop])
CALL axes[2].set_title("Bilinear")
CALL axes[3].imshow(bicubic[crop])
CALL axes[3].set_title("Bicubic")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "14_interpolation_comparison.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 21 — 20. Reflection / Flipping
(height, width) ← einstein.shape
F ← (build_translation_matrix((width − 1), 0) · build_horizontal_reflection_matrix())
reflected ← warp_image_affine(einstein, F, interpolation_order = 0)
reflected_numpy ← einstein[(:, ::(−1))]
(fig, axes) ← PLOT.subplots(1, 3, figsize = (13, 4))
CALL axes[0].imshow(einstein, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(reflected, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Homogeneous transform")
CALL axes[2].imshow(reflected_numpy, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("NumPy slicing")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "15_reflection.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Two reflection methods identical:", ARRAY.array_equal(reflected, reflected_numpy))

# Algorithm 22 — 21. Shear
shear_centered ← center_transform_on_image(build_shear_matrix(kx = 0.35, ky = 0.0), einstein.shape)
sheared ← warp_image_affine(einstein, shear_centered, interpolation_order = 1)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 4))
CALL axes[0].imshow(einstein, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(sheared, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Horizontal shear")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "16_shear.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 23 — 22. Composition of Transformations
translate ← build_translation_matrix(70, 20)
rotate ← center_transform_on_image(build_rotation_matrix(25), einstein.shape)
translate_then_rotate ← (rotate · translate)
rotate_then_translate ← (translate · rotate)
image_a ← warp_image_affine(einstein, translate_then_rotate, interpolation_order = 1)
image_b ← warp_image_affine(einstein, rotate_then_translate, interpolation_order = 1)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (13, 4))
CALL axes[0].imshow(einstein, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(image_a, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Translate → Rotate")
CALL axes[2].imshow(image_b, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("Rotate → Translate")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "17_transformation_order.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 24 — 23. General Affine Transformation
affine ← (((build_translation_matrix(35, (−10)) · center_transform_on_image(build_rotation_matrix((−18)),
    einstein.shape)) · center_transform_on_image(build_shear_matrix(kx = 0.18),
    einstein.shape)) · center_transform_on_image(build_scaling_matrix(0.88, 1.08), einstein.shape))
affine_result ← warp_image_affine(rgb_images["einstein"], affine, interpolation_order = 1)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 4.5))
CALL axes[0].imshow(rgb_images["einstein"])
CALL axes[0].set_title("Original")
CALL axes[1].imshow(affine_result)
CALL axes[1].set_title("Combined affine transform")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "18_affine_transform.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 25 — 24. Resizing to a New Array Shape
source ← IMAGE.fromarray(rgb_images["peppers"])
(original_width, original_height) ← source.size
new_width ← (original_width DIV 2)
new_height ← (original_height DIV 2)
resized_nearest ← ARRAY.asarray(source.resize((new_width, new_height), resample = IMAGE.Resampling.NEAREST))
resized_bilinear ← ARRAY.asarray(source.resize((new_width, new_height),
    resample = IMAGE.Resampling.BILINEAR))
resized_bicubic ← ARRAY.asarray(source.resize((new_width, new_height), resample = IMAGE.Resampling.BICUBIC))
CALL EMIT("Original shape :", rgb_images["peppers"].shape)
CALL EMIT("Resized shape  :", resized_bilinear.shape)
CALL EMIT("Aspect ratios  :", ROUND((original_width / original_height), 4), "→",
    ROUND((new_width / new_height), 4))

# Algorithm 26 — 25. Validation Checks
REQUIRE ARRAY.array_equal(identity, ascent) OTHERWISE STOP WITH ValidationError
REQUIRE (negative.dtype = ARRAY.uint8) OTHERWISE STOP WITH ValidationError
REQUIRE ((MIN(negative) ≥ 0) AND (MAX(negative) ≤ 255)) OTHERWISE STOP WITH ValidationError
REQUIRE (brighter.dtype = ARRAY.uint8) OTHERWISE STOP WITH ValidationError
gamma_identity ← apply_gamma_intensity_transform(ascent, 1.0)
REQUIRE ARRAY.array_equal(gamma_identity, ascent) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.all((ARRAY.diff(CAST(equalization_map, ARRAY.int16)) ≥ 0)) OTHERWISE STOP WITH ValidationError
REQUIRE (mapped_points.shape = source_points.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.allclose(mapped_points, expected_mapped_points) OTHERWISE STOP WITH ValidationError
REQUIRE (translated_image.shape = einstein.shape) OTHERWISE STOP WITH ValidationError
REQUIRE (rotated_center.shape = einstein.shape) OTHERWISE STOP WITH ValidationError
REQUIRE (affine_result.shape = rgb_images["einstein"].shape) OTHERWISE STOP WITH ValidationError
known_point ← transform_point_with_matrix(build_translation_matrix(12, (−5)), 10, 20)
REQUIRE ARRAY.allclose(known_point, [22, 15]) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.array_equal(reflected, reflected_numpy) OTHERWISE STOP WITH ValidationError
REQUIRE (NOT ARRAY.isclose(ARRAY.linalg.det(build_translation_matrix(10, 20)),
    0)) OTHERWISE STOP WITH ValidationError
REQUIRE (NOT ARRAY.isclose(ARRAY.linalg.det(build_rotation_matrix(30)),
    0)) OTHERWISE STOP WITH ValidationError
REQUIRE (NOT ARRAY.isclose(ARRAY.linalg.det(build_scaling_matrix(0.5, 1.5)),
    0)) OTHERWISE STOP WITH ValidationError
CALL EMIT("All image-transformation validation checks passed.")
REQUIRED_OUTPUTS ← ["01_reference_images.png", "02_negative.png", "03_brightness_contrast.png", "04_contrast_stretching.png",
    "05_log_transform.png", "06_gamma_examples.png", "07_gamma_curves.png", "08_histogram_equalization.png",
    "09_equalization_mapping.png", "10_intensity_transform_comparison.png", "10_geometric_coordinate_model.png",
    "11_translation.png", "12_rotation_origin_center.png", "13_scaling.png", "14_interpolation_comparison.png",
    "15_reflection.png", "16_shear.png", "17_transformation_order.png", "18_affine_transform.png"]
missing_outputs ← LIST[output_name FOR output_name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / output_name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT(FORMAT("Output validation passed: {0} files." , LENGTH(REQUIRED_OUTPUTS)))

END ALGORITHM
```

## Part II — Step-by-step auditable specification

Each step below is a six-section contract corresponding to the numbered block in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts and shapes required before execution.
- **Procedure** — ordered computation.
- **Produces** — outputs and invariants.
- **Acceptance contract** — numerical or structural gates before the next step.
- **Failure mode** — explicit abort, skip, or diagnostic action; never a silent substitution.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project environment.
2. Use this notebook as the exact coding order for `main.ipynb`.
3. Keep intensity transformations and geometric transformations as two clearly separated implementation parts.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Setup and Resolve Input Images

**Purpose**

Resolve common inputs for all transformation comparisons.

**Inputs**

Lab images and Python dependencies.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, PIL, and SciPy `map_coordinates`.
2. Locate the lab root and create data/output paths.
3. Discover required real images and map them to semantic roles.
4. Load RGB images and derive grayscale inputs needed by pointwise/geometric experiments.

**Produces**

RGB/grayscale dictionaries and output path.

**Acceptance contract**

Required roles load with expected channel dimensions.

**Failure mode**

Stop on a missing image role or invalid root.

## 2. Save the Reference-Image Overview

**Purpose**

Establish an unchanged visual baseline.

**Inputs**

Loaded reference images.

**Procedure**

1. Display the loaded reference images before any transformation.
2. Use correct RGB/grayscale display conventions.
3. Save the figure as the fixed baseline for later comparisons.

**Produces**

Reference overview PNG.

**Acceptance contract**

RGB/grayscale display conventions agree with arrays.

**Failure mode**

Fix wrong subjects or channel ordering before transforming.

## 3. Define the Intensity-Transformation Model

**Purpose**

Separate intensity arithmetic from geometry.

**Inputs**

8-bit image and pointwise mapping.

**Procedure**

1. Treat pointwise transforms as `s=T(r)`.
2. Convert to float or wider dtype before arithmetic when overflow is possible.
3. Clip results to display range before casting to uint8.
4. Keep identity cases for validation.

**Produces**

Identity baseline and intensity-transform convention.

**Acceptance contract**

Spatial coordinates stay unchanged; arithmetic uses a wider type.

**Failure mode**

Resolve overflow before evaluating appearance.

## 4. Implement Image Negative

**Purpose**

Reverse the 8-bit intensity scale.

**Inputs**

Grayscale uint8 image.

**Procedure**

1. Compute `255-image` for an 8-bit grayscale source.
2. Preserve shape and return uint8.
3. Display and save original versus negative.

**Produces**

negative=255-image.

**Acceptance contract**

Shape/dtype preserved; applying negative twice restores source.

**Failure mode**

Check data range if input is not 8-bit.

## 5. Implement Linear Brightness and Contrast

**Purpose**

Control contrast gain and brightness offset independently.

**Inputs**

Image, gain and offset.

**Procedure**

1. Create a helper implementing `a·image+b` in float arithmetic.
2. Use `a` as contrast gain and `b` as brightness offset.
3. Clip to `[0,255]` and cast to uint8.
4. Run representative brightness/contrast cases.

**Produces**

Clipped uint8 linear transformation.

**Acceptance contract**

gain=1 and offset=0 is identity; clipping explains saturation.

**Failure mode**

Promote input before arithmetic and diagnose saturation.

## 6. Implement Contrast Stretching

**Purpose**

Use the observed range for contrast stretching.

**Inputs**

Image minimum and maximum.

**Procedure**

1. Compute source minimum and maximum in float.
2. If the image is constant, return a safe constant result.
3. Map the observed range linearly to `[0,255]`.
4. Compare image and histogram before/after stretching.

**Produces**

Stretched image and histograms.

**Acceptance contract**

Nonconstant range maps to 0..255; constant input returns zeros.

**Failure mode**

Take the constant branch instead of dividing by zero.

## 7. Implement Logarithmic Transformation

**Purpose**

Expand dark intensities with a monotone mapping.

**Inputs**

8-bit grayscale image.

**Procedure**

1. Convert input to float.
2. Choose `c=255/log(256)` so maximum 8-bit input maps near 255.
3. Compute `c·log(1+r)`.
4. Clip/cast and save transformed image plus response curve.

**Produces**

Log-transformed image and response.

**Acceptance contract**

c=255/log(256); endpoints map near 0 and 255.

**Failure mode**

Reject incompatible ranges before applying log1p.

## 8. Implement Gamma Transformation

**Purpose**

Control nonlinear brightness with positive gamma.

**Inputs**

Image and gamma.

**Procedure**

1. Require strictly positive gamma.
2. Normalize intensities to `[0,1]`.
3. Apply `r**gamma`, rescale to `[0,255]`, clip, and cast.
4. Evaluate several gamma values including gamma=1 as identity.

**Produces**

Gamma-corrected uint8 image.

**Acceptance contract**

gamma>0; normalize to [0,1]; gamma=1 identity checked.

**Failure mode**

Raise for nonpositive gamma; investigate incorrect normalization.

## 9. Plot Gamma Response Curves

**Purpose**

Explain the gamma image results through their transfer curves.

**Inputs**

Same gamma values as image experiments.

**Procedure**

1. Create normalized input values from 0 to 1.
2. Plot `r**gamma` for the same gamma values used on images.
3. Include the identity line.
4. Use this plot to connect gamma choice to intensity response.

**Produces**

Curves on [0,1].

**Acceptance contract**

Curves share endpoints and include identity.

**Failure mode**

Use identical gamma definitions; do not interchange gamma and its reciprocal.

## 10. Implement Histogram Equalization

**Purpose**

Redistribute intensities through a cumulative histogram.

**Inputs**

8-bit grayscale array.

**Procedure**

1. Count all 256 intensity values.
2. Normalize counts to probabilities and compute cumulative distribution.
3. Map the CDF to `[0,255]` and use it as a lookup table.
4. Return both equalized image and mapping.

**Produces**

Equalized image and 256-entry lookup table.

**Acceptance contract**

Histogram counts sum to pixel count; mapping is nondecreasing.

**Failure mode**

Cast mapping to signed dtype before testing differences; no claim of an exactly uniform output histogram.

## 11. Compare Intensity Transformations

**Purpose**

Compare pointwise methods without display bias.

**Inputs**

Stored intensity-transform outputs.

**Procedure**

1. Create one grid containing original, negative, brightness/contrast, stretch, log, selected gamma results, and equalization.
2. Use identical grayscale limits.
3. Save the comparison as the summary of the intensity part.

**Produces**

Common comparison grid.

**Acceptance contract**

Same source and fixed display range for every panel.

**Failure mode**

Correct autoscaling or source mismatch before comparison.

## 12. Validate a Direct Coordinate Mapping

**Purpose**

Verify coordinate equations before image interpolation.

**Inputs**

Known points shaped (N,2).

**Procedure**

1. Create known 2-D source points.
2. Apply explicit scale-and-shift equations to x/y coordinates.
3. Compute expected transformed coordinates independently.
4. Assert numerical equality before image warping.
5. Save source-versus-mapped coordinate figure.

**Produces**

Mapped points and geometry illustration.

**Acceptance contract**

Independent expected scale/shift values match with allclose.

**Failure mode**

Correct x/y order or transform signs before warping.

## 13. Implement Homogeneous Point Transformation

**Purpose**

Represent point geometry with matrix products.

**Inputs**

3 by 3 matrix and point (x,y).

**Procedure**

1. Represent each 2-D point as `[x,y,1]`.
2. Multiply by a 3×3 transform matrix.
3. Divide the first two coordinates by the homogeneous third coordinate.
4. Verify identity returns the original point.

**Produces**

Dehomogenized 2-D point.

**Acceptance contract**

Identity preserves point; translation matches a known example.

**Failure mode**

Do not divide by zero homogeneous coordinate; affine matrices have last row [0,0,1].

## 14. Build Fundamental 3×3 Transform Matrices

**Purpose**

Build composable affine primitives.

**Inputs**

Offsets, scale factors, angle and shear.

**Procedure**

1. Implement translation from `(tx,ty)`.
2. Implement scaling from `(sx,sy)`.
3. Implement rotation after converting degrees to radians.
4. Implement shear and horizontal reflection.
5. Keep every transform in homogeneous-coordinate form so matrices can be composed.

**Produces**

3 by 3 translation, scale, rotation, shear/reflection matrices.

**Acceptance contract**

Angles converted to radians; last row [0,0,1]; transforms used for inverse warping are invertible.

**Failure mode**

Reject singular parameters such as zero scale before inversion.

## 15. Center a Transform on the Image

**Purpose**

Rotate or scale about the image centre.

**Inputs**

Affine matrix and image shape.

**Procedure**

1. Compute image center `(width-1)/2,(height-1)/2`.
2. Build translation to move the center to origin.
3. Apply the requested transform.
4. Translate back with `T_from_origin @ M @ T_to_origin`.

**Produces**

T_from_origin @ M @ T_to_origin.

**Acceptance contract**

Centre uses ((width-1)/2,(height-1)/2); a centred linear transform fixes it.

**Failure mode**

Correct composition order or row/column conventions.

## 16. Implement Inverse-Mapped Affine Warping

**Purpose**

Avoid holes by sampling each destination pixel.

**Inputs**

Image, invertible forward affine matrix, output shape and interpolation order.

**Procedure**

1. Create output-grid coordinates for every destination pixel.
2. Invert the forward transform once.
3. Map destination coordinates back to source coordinates.
4. Sample source values with `map_coordinates` at the requested interpolation order.
5. Process RGB channels separately.
6. Clip/cast after interpolation.

**Produces**

Warped uint8 image.

**Acceptance contract**

Apply inverse once; sample_coordinates is [y_in,x_in]; process RGB channels separately.

**Failure mode**

Stop on singular matrix/unsupported dimensions; document constant fill outside the source.

## 17. Apply Translation

**Purpose**

Move image content by a known displacement.

**Inputs**

Translation offsets and source image.

**Procedure**

1. Build a translation matrix from declared offsets.
2. Warp with nearest-neighbour interpolation for the direct translation demo.
3. Verify output shape and save the result.

**Produces**

Translated image.

**Acceptance contract**

Same canvas shape; nearest-neighbour sampling in this demo.

**Failure mode**

Distinguish expected clipping at the canvas from incorrect displacement.

## 18. Compare Rotation About Origin and Center

**Purpose**

Show why the rotation centre matters.

**Inputs**

One angle and common source.

**Procedure**

1. Build one rotation about coordinate origin.
2. Build the same rotation centered on the image.
3. Warp both using identical interpolation.
4. Save original, origin-rotation, and center-rotation together.

**Produces**

Origin and centre rotation results.

**Acceptance contract**

Angle/interpolation identical; only pivot differs.

**Failure mode**

Fix pivot construction before attributing differences to interpolation.

## 19. Apply Uniform and Non-Uniform Scaling

**Purpose**

Separate scaling from canvas resizing.

**Inputs**

Source and uniform/anisotropic factors.

**Procedure**

1. Build centered scale matrices for the declared factors.
2. Apply inverse warping.
3. Compare uniform with anisotropic scaling.
4. Keep output canvas shape fixed for this experiment.

**Produces**

Scaled images on the original-size canvas.

**Acceptance contract**

Matrix factors differ as intended; output shape remains fixed.

**Failure mode**

Explain clipping/padding rather than claiming array dimensions changed.

## 20. Compare Interpolation Methods

**Purpose**

Isolate interpolation effects.

**Inputs**

One transform, source and common crop.

**Procedure**

1. Apply exactly the same scale with nearest, bilinear, and bicubic interpolation.
2. Inspect the same crop from every result.
3. Save the comparison so only interpolation changes.

**Produces**

Nearest, bilinear and bicubic comparisons.

**Acceptance contract**

Only interpolation order changes; crop coordinates identical.

**Failure mode**

Correct a changed transform/canvas before assessing sharpness.

## 21. Implement Horizontal Reflection

**Purpose**

Verify reflection against an independent array operation.

**Inputs**

Source and reflection plus translation matrix.

**Procedure**

1. Build reflection about the y-axis in homogeneous coordinates.
2. Compose the translation required to keep coordinates inside the image canvas.
3. Warp with nearest-neighbour interpolation.
4. Compare against NumPy `image[:,::-1]` and assert exact equality.

**Produces**

Reflected image.

**Acceptance contract**

Nearest-neighbour result equals image[:,::-1] exactly.

**Failure mode**

Correct translation by width-1 or axis convention.

## 22. Apply Shear

**Purpose**

Create a controlled shear example.

**Inputs**

Source and centred shear matrix.

**Procedure**

1. Build the declared horizontal shear matrix.
2. Center it on the image.
3. Warp with bilinear interpolation.
4. Save original and sheared result.

**Produces**

Sheared image.

**Acceptance contract**

Bilinear interpolation; matrix is invertible.

**Failure mode**

Check shear direction and determinant before inverse mapping.

## 23. Demonstrate Non-Commutative Composition

**Purpose**

Demonstrate that composition order changes geometry.

**Inputs**

Translation and centred rotation.

**Procedure**

1. Build translation and centered rotation matrices.
2. Compute `rotate @ translate` and `translate @ rotate` separately.
3. Warp the same image with both matrices.
4. Save the comparison to show order changes the result.

**Produces**

Two differently composed warped images.

**Acceptance contract**

Rightmost matrix acts first on column coordinates.

**Failure mode**

Correct the description if visual order and multiplication order disagree.

## 24. Build a General Affine Transform

**Purpose**

Apply a composite transform in one resampling pass.

**Inputs**

Translation, rotation, shear and scaling matrices.

**Procedure**

1. Compose translation, centered rotation, centered shear, and centered scaling in the declared order.
2. Apply the composite matrix to the RGB image.
3. Use bilinear interpolation.
4. Save original versus combined affine result.

**Produces**

RGB affine result.

**Acceptance contract**

Declared order and source channels preserved; composite invertible.

**Failure mode**

Inspect matrices before interpolation; avoid repeated resampling disguised as one transform.

## 25. Resize to a New Array Shape

**Purpose**

Change array size explicitly.

**Inputs**

PIL image and chosen output width/height.

**Procedure**

1. Convert a representative RGB array to PIL.
2. Choose new width/height while preserving intended aspect ratio.
3. Resize with nearest, bilinear, and bicubic methods.
4. Print original/resized shapes and aspect ratios.

**Produces**

Resized arrays for three interpolation modes.

**Acceptance contract**

PIL size uses (width,height); NumPy shape uses (height,width,channels).

**Failure mode**

Correct swapped dimensions or unintended aspect-ratio changes.

## 26. Run Final Validation

**Purpose**

Validate intensity identities and geometric consistency.

**Inputs**

Computed transforms, arrays and required files.

**Procedure**

1. Verify intensity identity cases, dtype/range constraints, and monotonic equalization mapping.
2. Verify coordinate-mapping test points and homogeneous translation test.
3. Verify warped shapes and exact reflection equivalence.
4. Verify transform matrices used here are invertible.
5. Verify all 19 declared output files before printing the final pass message.

**Produces**

Final validation outcome.

**Acceptance contract**

Identity/gamma/reflection checks pass; mapping monotone; known points match; matrices invertible; 19 PNGs exist.

**Failure mode**

Stop on failed assertions or missing figures; separate canvas clipping from numerical failure.